# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdullah-pharmd/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

### Plain-Words Data Contract: Unit of Analysis and Time Window

1. **Unit of Analysis (The Grain)**: 
   One row represents **one unique content item (web page) for a specific client**, summarizing its daily Google Search Console (GSC) visibility and Google Analytics 4 (GA4) engagement metrics over a specific calendar month. In our database, this grain is identified by the composite key `client_hash_id + content_hash_id`.

2. **Table(s) to Use**: 
   We will use the gated Hugging Face warehouse fact table: `fact_content_daily_performance` (partitioned by month).

3. **Time Window**: 
   We will use a mid-panel month partition: **March 2026** (`month=2026-03` partition, spanning `2026-03-01` to `2026-03-31` inclusive) for our feature window, and we will split it into a feature-building period (March 1-20) and a target-monitoring period (March 21-31).

4. **What we predict or rank (Label / Proxy)**: 
   We predict a binary **traffic decline label** (`is_declining`), defined as a **greater than 20% drop in average daily GSC impressions** in the late-month window (March 21-31) compared to the early-month baseline window (March 1-20).

5. **Deliberate Exclusion**: 
   We deliberately exclude `trend_pct`, `trend_direction`, and `is_declining_label` from the feature window, as well as the late-window impressions (`imp_late`). These contain information about the outcome period, and including them in the feature set would create **target leakage**, artificially inflating model performance while making it useless in production.

In [1]:
# Documenting our data contract parameters
print("Contract Grain: client_hash_id + content_hash_id")
print("Contract Table: fact_content_daily_performance (month=2026-03)")
print("Contract Target: is_declining (late-month daily impressions drop > 20%)")


Contract Grain: client_hash_id + content_hash_id
Contract Table: fact_content_daily_performance (month=2026-03)
Contract Target: is_declining (late-month daily impressions drop > 20%)


## 2. Fields: feature / label / context / excluded

### Field Classification Table

| Field Name | Classification | Meaning | Rationale / Excluded Why | 
|---|---|---|---|
| `client_hash_id` | Context | Scrambled client identifier | Used for grouped splits (client holdout) to check generalization; never a feature |
| `content_hash_id` | Context | Scrambled page identifier | Used for joining and tracking; never a feature |
| `imp_early` | Feature | Sum of GSC impressions (March 1-20) | Safe feature: represents search visibility in the early-month window |
| `clk_early` | Feature | Sum of GSC clicks (March 1-20) | Safe feature: represents search traffic volume in the early-month window |
| `pos_early` | Feature | Average GSC position (March 1-20) | Safe feature: represents average search ranking position |
| `ses_early` | Feature | Sum of GA4 sessions (March 1-20) | Safe feature: represents on-site user session volume |
| `ctr_early` | Feature | Click-Through Rate (March 1-20) | Safe feature: derived clicks / impressions in the feature window |
| `is_declining` | Label | Binary decline target | Computed from late-month impressions compared to early-month impressions |
| `imp_late` | Excluded | Sum of GSC impressions (March 21-31) | **Excluded**: This contains the exact target-window visibility data. Using it as a feature is **direct target leakage** |
| `trend_pct` / `trend_direction` | Excluded | Built-in trend indicators | **Excluded**: They represent the outcome labels in the starter slice and would cause circular reasoning if used as features |
| `health_score` / `priority_score` | Excluded | Hand-tuned product rules | **Excluded**: These represent existing product decisions. The model should find raw patterns, not memorize old rules |

In [2]:
# Displaying our feature classifications
features = ['imp_early', 'clk_early', 'pos_early', 'ses_early', 'ctr_early']
excluded = ['imp_late', 'trend_pct', 'trend_direction', 'health_score', 'priority_score']
print("Model Features:", features)
print("Excluded Features (Leakage Risks/Product Flags):", excluded)


Model Features: ['imp_early', 'clk_early', 'pos_early', 'ses_early', 'ctr_early']
Excluded Features (Leakage Risks/Product Flags): ['imp_late', 'trend_pct', 'trend_direction', 'health_score', 'priority_score']


## 3. Verify it with queries (grain, counts, missing values, windows)

Here, we connect to the Hugging Face warehouse using DuckDB and run our verification queries on the March 2026 partition. We also execute the leakage experiment to demonstrate the impact of target leakage.

In [3]:
import os, getpass
import duckdb
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

# Authenticate using HF token
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    # Try reading from local .env file
    try:
        if os.path.exists('../../.env'):
            with open('../../.env', 'r') as f:
                for line in f:
                    if line.strip().startswith('HF_TOKEN='):
                        HF_TOKEN = line.strip().split('=', 1)[1]
        elif os.path.exists('.env'):
            with open('.env', 'r') as f:
                for line in f:
                    if line.strip().startswith('HF_TOKEN='):
                        HF_TOKEN = line.strip().split('=', 1)[1]
    except Exception:
        pass
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

# =========================================================================
# PART TWO: Verification Queries
# =========================================================================

# Query 1: Verify the grain (report_date + client_hash_id + content_hash_id is unique)
print("Query 1: Verifying Grain Unique Constraint...")
grain_df = con.sql(f"""
    WITH grain_check AS (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as c
        FROM read_parquet('{FACT_PATH}')
        WHERE ga4_data_available IS TRUE
        GROUP BY 1, 2, 3
    )
    SELECT * FROM grain_check WHERE c > 1 LIMIT 5
""").df()
print(f"  Number of duplicate grain rows found: {len(grain_df)}")
assert len(grain_df) == 0, "Grain check failed! Duplicate rows exist."

# Query 2: Slice's row count and date span
print("\nQuery 2: Fetching Row Count and Date Span for March 2026...")
span_df = con.sql(f"""
    SELECT 
        COUNT(*) as total_daily_rows,
        COUNT(DISTINCT client_hash_id) as distinct_clients,
        COUNT(DISTINCT content_hash_id) as distinct_content_items,
        MIN(report_date) as min_report_date,
        MAX(report_date) as max_report_date
    FROM read_parquet('{FACT_PATH}')
""").df()
print(span_df.to_string(index=False))

# Query 3: Availability (checked with IS TRUE)
print("\nQuery 3: Verifying GA4 Data Availability...")
avail_df = con.sql(f"""
    SELECT 
        COUNT(*) as total_daily_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available_rows,
        AVG(CASE WHEN ga4_data_available IS TRUE THEN 1.0 ELSE 0.0 END) * 100 as ga4_available_pct
    FROM read_parquet('{FACT_PATH}')
""").df()
print(avail_df.to_string(index=False))

# =========================================================================
# PART THREE: Feature Frame Construction
# =========================================================================
print("\nBuilding 5-Feature Frame (March 1-20 features, March 21-31 outcome)...")
data_df = con.sql(f"""
    WITH base AS (
        SELECT 
            client_hash_id,
            content_hash_id,
            -- Features (March 1-20)
            SUM(CASE WHEN report_date <= '2026-03-20' THEN gsc_impressions ELSE 0 END) as imp_early,
            SUM(CASE WHEN report_date <= '2026-03-20' THEN gsc_clicks ELSE 0 END) as clk_early,
            AVG(CASE WHEN report_date <= '2026-03-20' THEN gsc_avg_position END) as pos_early,
            SUM(CASE WHEN report_date <= '2026-03-20' THEN ga4_sessions ELSE 0 END) as ses_early,
            -- Outcome target (March 21-31)
            SUM(CASE WHEN report_date > '2026-03-20' THEN gsc_impressions ELSE 0 END) as imp_late
        FROM read_parquet('{FACT_PATH}')
        WHERE ga4_data_available IS TRUE
        GROUP BY 1, 2
        HAVING imp_early >= 50 -- Filter out low volume noise
    )
    SELECT 
        client_hash_id,
        content_hash_id,
        imp_early,
        clk_early,
        pos_early,
        ses_early,
        (clk_early / NULLIF(imp_early, 0)) * 100 as ctr_early,
        imp_late,
        -- is_declining target: avg daily impressions late < 0.8 * avg daily impressions early
        -- (imp_late / 11) < 0.8 * (imp_early / 20) => imp_late < 0.44 * imp_early
        CASE WHEN imp_late < 0.44 * imp_early THEN 1 ELSE 0 END as is_declining
    FROM base
""").df()

# Handle missing data safely
data_df['pos_early'] = data_df['pos_early'].fillna(0)
data_df['ctr_early'] = data_df['ctr_early'].fillna(0)

print(f"DataFrame constructed: {len(data_df):,} content items.")
print(data_df[['imp_early', 'clk_early', 'pos_early', 'ses_early', 'ctr_early', 'is_declining']].head(3))

# =========================================================================
# PART FOUR: Target Leakage Experiment
# =========================================================================
honest_features = ['imp_early', 'clk_early', 'pos_early', 'ses_early', 'ctr_early']
X_honest = data_df[honest_features]
y = data_df['is_declining']

X_train_h, X_test_h, y_train, y_test = train_test_split(X_honest, y, test_size=0.25, random_state=42, stratify=y)

# 1. Train Honest Model
clf_honest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf_honest.fit(X_train_h, y_train)
y_pred_h = clf_honest.predict_proba(X_test_h)[:, 1]
honest_auc = roc_auc_score(y_test, y_pred_h)
print(f"\n[Honest Model] ROC-AUC: {honest_auc:.4f}")

# 2. Train Leaky Model (The Trap - adding imp_late which contains future target data)
leaky_features = honest_features + ['imp_late']
X_leaky = data_df[leaky_features]

X_train_l, X_test_l, _, _ = train_test_split(X_leaky, y, test_size=0.25, random_state=42, stratify=y)
clf_leaky = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf_leaky.fit(X_train_l, y_train)
y_pred_l = clf_leaky.predict_proba(X_test_l)[:, 1]
leaky_auc = roc_auc_score(y_test, y_pred_l)
print(f"[Leaky Model] ROC-AUC: {leaky_auc:.4f}  <-- DANGER: Target Leakage makes the model look perfect!")

print("\nLeakage experiment complete. We will delete/ignore the leaky features and deploy only the honest model.")


Query 1: Verifying Grain Unique Constraint...


  Number of duplicate grain rows found: 0

Query 2: Fetching Row Count and Date Span for March 2026...


 total_daily_rows  distinct_clients  distinct_content_items min_report_date max_report_date
          9841378                55                  331437      2026-03-01      2026-03-31

Query 3: Verifying GA4 Data Availability...


 total_daily_rows  ga4_available_rows  ga4_available_pct
          9841378            413966.0           4.206382

Building 5-Feature Frame (March 1-20 features, March 21-31 outcome)...


DataFrame constructed: 26,025 content items.
   imp_early  clk_early  pos_early  ses_early  ctr_early  is_declining
0     1426.0       65.0  27.954086       59.0   4.558205             0
1      157.0        1.0   3.503185        1.0   0.636943             1
2      101.0        2.0   3.851461        2.0   1.980198             1



[Honest Model] ROC-AUC: 0.6329


[Leaky Model] ROC-AUC: 0.9986  <-- DANGER: Target Leakage makes the model look perfect!

Leakage experiment complete. We will delete/ignore the leaky features and deploy only the honest model.


## 4. Data limits

### Primary Data Limitation: Unbalanced Panel and Varying Client History

One major limitation of our data slice is that it is an **unbalanced panel**. 

#### Description of the Limit:
Different clients joined the FlyRank platform at different times, meaning their historical data ranges vary wildly (some have 17 months of history, others have only 3 months). Furthermore, a client's GSC tracking and GA4 tracking often start on different dates. 

#### The Impact:
If we define a global calendar window (like "March 2026") for all clients, we will drop clients whose tracking had not started yet or whose history had already ended, leading to selection bias. To work honestly, we cannot use static calendar windows across the entire database; we must write queries that define windows **relative to each client's individual GSC/GA4 start dates** (`dim_clients.gsc_data_start` and `ga4_data_start`).

In [4]:
# Show that history start dates differ per client
clients_df = con.sql(f"""
    SELECT client_hash_id, gsc_data_start, ga4_data_start
    FROM read_parquet('{REL}/dim_clients.parquet')
    ORDER BY gsc_data_start NULLS LAST
    LIMIT 5
""").df()
print("Varying Client History Starts:")
print(clients_df)


Varying Client History Starts:
            client_hash_id gsc_data_start ga4_data_start
0  client_9958f0a7ae1df715     2025-01-27     2025-10-29
1  client_ff644d8251367cbb     2025-01-27     2025-10-29
2  client_73cda7b4e4f265ea     2025-02-11     2026-03-24
3  client_fef1a8f436438636     2025-03-11     2026-03-06
4  client_62f4a7e64f5e0096     2025-06-07            NaT


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.